# AGENTS_030 — Change Impact Agent (Hackathon Demo)

This notebook:
1. Uses **this local TheRock repo** (no clone) and **fetches upstream** `ROCm/TheRock` for PR analysis
2. Installs Python dependencies
3. Runs unit tests (`pytest`)
4. Analyzes real upstream PRs on `ROCm/TheRock`
5. Generates executive summaries and displays HTML reports (including **Topology warnings** — always shown, even when empty)

**Pipeline:** `analyze.py` is fully deterministic (topology graph + `topology_audit.py`). `summarize.py` uses `--backend template` by default; optional LLM backends apply validation guardrails.

**Secrets:** paste `GITHUB_TOKEN` in `agents/change-impact-agent/.env` (gitignored) — never hardcode tokens in this notebook.

**Optional:** set `THEROCK_ROOT` if the repo is not auto-detected from the working directory.

**Out of scope:** ARVIL integration, cross-repo pattern scanning (see `SCOPE.md` in the agent folder).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# --- Configuration (edit if needed) ---
BRANCH = os.environ.get("THEROCK_BRANCH", "feature/change-impact-agent")
UPSTREAM_REPO = "ROCm/TheRock"
UPSTREAM_GIT = f"https://github.com/{UPSTREAM_REPO}.git"
# PRs used in the hackathon demo (upstream ROCm/TheRock)
DEMO_PRS = [5572, 5688, 5480]


def resolve_therock_root() -> Path:
    """Find TheRock repo root from THEROCK_ROOT or by walking up from cwd."""
    env_root = os.environ.get("THEROCK_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / ".git").is_dir() and (root / "agents" / "change-impact-agent" / "analyze.py").is_file():
            return root
        raise RuntimeError(f"THEROCK_ROOT is not a TheRock git repo: {root}")

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / ".git").is_dir()
            and (candidate / "agents" / "change-impact-agent" / "analyze.py").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Could not find TheRock repo root. Start Jupyter from the repo or set THEROCK_ROOT."
    )


def read_env_file(env_path: Path) -> dict[str, str]:
    """Read KEY=VALUE pairs directly from a .env file (no shell export required)."""
    if not env_path.is_file():
        return {}
    values: dict[str, str] = {}
    for raw_line in env_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[7:].strip()
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            values[key] = value
    return values


def apply_env_file(env_path: Path) -> dict[str, str]:
    """Load .env into this Jupyter kernel and os.environ for child processes."""
    values = read_env_file(env_path)
    for key, value in values.items():
        if os.environ.get(key) in (None, ""):
            os.environ[key] = value
    return values


REPO = resolve_therock_root()
AGENT = REPO / "agents" / "change-impact-agent"
ENV_FILE = AGENT / ".env"

sys.path.insert(0, str(AGENT))

AGENT_ENV = read_env_file(ENV_FILE)
apply_env_file(ENV_FILE)
GITHUB_TOKEN = (
    AGENT_ENV.get("GITHUB_TOKEN")
    or AGENT_ENV.get("GH_TOKEN")
    or os.environ.get("GITHUB_TOKEN")
    or os.environ.get("GH_TOKEN")
)
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
    print(f"GITHUB_TOKEN loaded from {ENV_FILE}")
else:
    print(f"GITHUB_TOKEN not set — create {ENV_FILE} (see .env.example)")

print(f"TheRock root: {REPO}")
print(f"Branch: {BRANCH}")

## 1. Sync local repo + fetch upstream

Uses the existing git repo (no clone). Fetches your `origin` branch and upstream `ROCm/TheRock` `main` as `upstream-main` for PR merge-base analysis.

In [ ]:
def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode, cmd, output=result.stdout, stderr=result.stderr
        )
    return result


def fetch_upstream_main(depth: int = 200) -> None:
    """Fetch ROCm/TheRock main into local ref upstream-main (no clone)."""
    run(
        ["git", "fetch", UPSTREAM_GIT, f"main:upstream-main", f"--depth={depth}"],
        cwd=REPO,
        check=False,
    )


print(f"Using local repo: {REPO}")
run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=False)
run(["git", "checkout", BRANCH], cwd=REPO, check=False)
fetch_upstream_main()

AGENT = REPO / "agents" / "change-impact-agent"
OUT = AGENT / "out"
apply_env_file(AGENT / ".env")
if os.environ.get("GITHUB_TOKEN"):
    print(f"GITHUB_TOKEN available for subprocesses (from {AGENT / '.env'})")
else:
    print(f"No GITHUB_TOKEN — create {AGENT / '.env'} (see .env.example)")

import json


def summarize_cli_supports_validation() -> bool:
    help_text = subprocess.run(
        [sys.executable, str(AGENT / "summarize.py"), "--help"],
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    return "--validate-llm" in (help_text.stdout + help_text.stderr)


def configure_vllm_client_env(
    base_url: str = "http://localhost:8000/v1",
    api_key: str = "abc-123",
    model: str = "Qwen3-30B-A3B",
) -> tuple[str, str, str]:
    """AMD AI Agents 101 vLLM workshop env pattern (OpenAI-compatible endpoint)."""
    os.environ["VLLM_BASE_URL"] = base_url
    os.environ["BASE_URL"] = base_url
    os.environ["OPENAI_API_KEY"] = api_key
    os.environ["VLLM_MODEL"] = model
    os.environ["SUMMARY_LLM_BACKEND"] = "vllm"
    return base_url, api_key, model


def verify_vllm_server(base_url: str, api_key: str) -> bool:
    import urllib.error
    import urllib.request

    url = f"{base_url.rstrip('/')}/models"
    req = urllib.request.Request(
        url,
        headers={"Authorization": f"Bearer {api_key}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            body = resp.read().decode("utf-8", errors="replace")
            print(body[:800])
            if len(body) > 800:
                print("...")
            return True
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        print(f"vLLM not reachable at {url}: {exc}")
        return False


def print_topology_warnings(report_path: Path) -> None:
    """Mirror report.html / executive_summary topology section."""
    if not report_path.is_file():
        return
    report = json.loads(report_path.read_text(encoding="utf-8"))
    warnings = report.get("topology_warnings") or []
    print("Topology warnings:")
    if warnings:
        for warning in warnings:
            print(f"  - {warning}")
    else:
        print("  - No topology gaps detected for this change range.")


def summarize_pr_output(
    pr_out: Path,
    backend: str = "template",
    *,
    base_url: str | None = None,
    model: str | None = None,
    output_name: str = "executive_summary.md",
) -> str:
    """Run summarize.py (template or LLM backend with validation guardrails)."""
    output_path = pr_out / output_name
    cmd = [
        sys.executable,
        str(AGENT / "summarize.py"),
        "--backend",
        backend,
        "--input",
        str(pr_out / "report.json"),
        "--output",
        str(output_path),
    ]
    if base_url:
        cmd.extend(["--base-url", base_url])
    if model:
        cmd.extend(["--model", model])
    if backend != "template" and summarize_cli_supports_validation():
        cmd.extend(["--validate-llm", "--fallback-template"])
    run(cmd, cwd=REPO)
    print_topology_warnings(pr_out / "report.json")
    return output_path.read_text(encoding="utf-8")


def run_llm_executive_summary(
    pr_out: Path,
    *,
    backend: str = "vllm",
    base_url: str | None = None,
    model: str | None = None,
    output_name: str = "executive_summary_llm.md",
) -> str | None:
    """Generate LLM prose summary; keeps template executive_summary.md unchanged."""
    apply_env_file(AGENT / ".env")
    report_path = pr_out / "report.json"
    if not report_path.is_file():
        print(f"No report at {report_path} — run section 5 first.")
        return None

    if backend == "openai":
        if not os.environ.get("OPENAI_API_KEY"):
            print("OPENAI_API_KEY not set — add to .env or skip this section.")
            return None
        base_url = base_url or "https://api.openai.com/v1"
        model = model or os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
    elif backend == "vllm":
        base_url = base_url or os.environ.get(
            "VLLM_BASE_URL", os.environ.get("BASE_URL", "http://localhost:8000/v1")
        )
        model = model or os.environ.get("VLLM_MODEL", "Qwen3-30B-A3B")
        os.environ.setdefault("OPENAI_API_KEY", "abc-123")
    else:
        raise ValueError(f"Unsupported LLM backend: {backend}")

    print(f"LLM summary: backend={backend}, model={model}, url={base_url}")
    try:
        return summarize_pr_output(
            pr_out,
            backend=backend,
            base_url=base_url,
            model=model,
            output_name=output_name,
        )
    except subprocess.CalledProcessError as exc:
        print(
            f"LLM summary failed (exit {exc.returncode}). "
            "Template executive_summary.md from summarize step is unchanged."
        )
        return None


def run_llm_summaries_for_demo_prs(
    backend: str = "vllm",
    *,
    base_url: str | None = None,
    model: str | None = None,
    prs: list[int] | None = None,
) -> dict[int, str | None]:
    """Generate executive_summary_llm.md for each demo PR (keeps template summaries)."""
    results: dict[int, str | None] = {}
    for pr in prs or DEMO_PRS:
        pr_out = OUT / f"pr-{pr}"
        print(f"\n{'='*60}\nLLM summary PR #{pr}\n{'='*60}")
        results[pr] = run_llm_executive_summary(
            pr_out,
            backend=backend,
            base_url=base_url,
            model=model,
            output_name="executive_summary_llm.md",
        )
        if results[pr]:
            preview = "\n".join(results[pr].splitlines()[:12])
            print(preview)
            if len(results[pr].splitlines()) > 12:
                print("...")
        else:
            print(f"No LLM summary for PR #{pr} — template executive_summary.md unchanged.")
    return results


print(f"TheRock root: {REPO}")
print(f"Agent: {AGENT}")

## 2. Install dependencies

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(AGENT / "requirements.txt")])
run([sys.executable, "-m", "pip", "install", "-q", "pytest"])

## 3. Run unit tests

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "agents/change-impact-agent/tests/", "-q"],
    cwd=REPO,
)
if result.returncode != 0:
    raise RuntimeError("pytest failed — fix tests before demo")
print("All unit tests passed.")

## 4. Analyze local range + executive summary

Step 1: `analyze.py` → `report.json` + `report.html`. Step 2: `summarize.py --backend template` → `executive_summary.md`.

Uses `HEAD~6..HEAD` on the checked-out branch in your local repo.

In [ ]:
demo_out = OUT / "demo-main-range"
run([
    sys.executable, str(AGENT / "analyze.py"),
    "--start", "HEAD~6", "--end", "HEAD",
    "--output-dir", str(demo_out),
], cwd=REPO)
print(summarize_pr_output(demo_out))

## 5. Analyze upstream PRs + executive summary (ROCm/TheRock)

Each PR: `analyze.py` then `summarize.py --backend template` → `executive_summary.md`. Optional LLM prose in section 9 → `executive_summary_llm.md` per PR. Section 1 fetches `upstream-main` — re-run section 1 if PR analysis fails.

| PR | Type |
|----|------|
| 5572 | MIOpen GHA timeout 60→120 min |
| 5688 | hipDNN CI + artifact TOML |
| 5480 | OpenMPI CMake version file naming |

PR #5718 (rocm-libraries superrepo bump) is omitted — slow full-manifest analysis.

In [ ]:
def ensure_upstream_main() -> None:
    """Fork clones may lack local main — fetch upstream tip as upstream-main."""
    fetch_upstream_main()

def analyze_supports_pr_flag() -> bool:
    help_text = subprocess.run(
        [sys.executable, str(AGENT / "analyze.py"), "--help"],
        cwd=REPO, capture_output=True, text=True,
    )
    return "--pr" in (help_text.stdout + help_text.stderr)

def analyze_upstream_pr(pr: int, pr_out: Path) -> int:
    """Analyze an upstream ROCm/TheRock PR using the local repo + upstream fetch."""
    ensure_upstream_main()
    if analyze_supports_pr_flag():
        return run([
            sys.executable, str(AGENT / "analyze.py"),
            "--pr", str(pr),
            "--full-manifest",
            "--output-dir", str(pr_out),
        ], cwd=REPO, check=False).returncode
    local_ref = f"pr-{pr}"
    run(["git", "fetch", UPSTREAM_GIT, f"pull/{pr}/head:{local_ref}"], cwd=REPO, check=False)
    return run([
        sys.executable, str(AGENT / "analyze.py"),
        "--end", local_ref,
        "--pr-base-ref", "upstream-main",
        "--output-dir", str(pr_out),
    ], cwd=REPO, check=False).returncode

pr_results = {}
for pr in DEMO_PRS:
    pr_out = OUT / f"pr-{pr}"
    print(f"\n{'='*60}\nPR #{pr}\n{'='*60}")
    rc = analyze_upstream_pr(pr, pr_out)
    if rc != 0:
        print(f"Warning: analyze failed for PR #{pr} (exit {rc})")
        continue
    summary = summarize_pr_output(pr_out)
    pr_results[pr] = summary
    # Print first 25 lines of each summary
    print("\n".join(summary.splitlines()[:25]))
    if len(summary.splitlines()) > 25:
        print("...")

## 6. List open upstream PRs (optional, no analyze)

Requires `GITHUB_TOKEN` in `agents/change-impact-agent/.env` (GitHub rate-limits anonymous API calls).

In [ ]:
apply_env_file(AGENT / ".env")
if not os.environ.get("GITHUB_TOKEN"):
    print(
        "Warning: no GITHUB_TOKEN — create agents/change-impact-agent/.env "
        "(copy from .env.example) or section 6 may hit GitHub rate limits."
    )
rc = run([
    sys.executable, str(AGENT / "upstream_pr_scan.py"),
    "--max", "5",
], cwd=REPO, check=False).returncode
if rc != 0:
    print(f"upstream_pr_scan failed (exit {rc}). Add GITHUB_TOKEN to {AGENT / '.env'}")

## 7. View HTML report (example: PR #5572)

In [ ]:
from IPython.display import HTML

example_pr = 5572
html_path = OUT / f"pr-{example_pr}" / "report.html"
if html_path.exists():
    HTML(html_path.read_text(encoding="utf-8"))
else:
    print(f"No report at {html_path} — run section 5 first")

## 9. LLM executive summary — vLLM on MI300 (AMD workshop pattern)

Runs after section 5 (`summarize.py` template step). For **each demo PR** in `DEMO_PRS`, writes `out/pr-<N>/executive_summary_llm.md` without replacing `executive_summary.md`.

**Reference:** [AMD AI Agents 101 — vLLM + MCP workshop notebook](https://github.com/sarimahsan/amd-ai-learning/blob/main/AI%20Agents%20101%20Building%20AI%20Agents%20with%20MCP%20and%20Open-Source%20Inference/build_airbnb_agent_mcp.ipynb) (same `BASE_URL` / `OPENAI_API_KEY` pattern).

### 9a. Launch vLLM in a terminal (MI300)

Open a **new terminal** on the GPU node (not this notebook kernel) and run:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
  --served-model-name Qwen3-30B-A3B \
  --api-key abc-123 \
  --port 8000 \
  --trust-remote-code
```

Optional: `watch rocm-smi` in another terminal to monitor GPU use.

### 9b. Run the cell below

Configures `BASE_URL=http://localhost:8000/v1` and `OPENAI_API_KEY=abc-123` like the workshop, verifies `/v1/models`, then calls `summarize.py --backend vllm` for PRs #5572, #5688, #5480.

**OpenAI instead of vLLM:** set `SUMMARY_LLM_BACKEND=openai` and `OPENAI_API_KEY` in `.env`.

In [ ]:
apply_env_file(AGENT / ".env")

LLM_BACKEND = os.environ.get("SUMMARY_LLM_BACKEND", "vllm")  # vllm | openai
pr_llm_results: dict[int, str | None] = {}

if LLM_BACKEND == "vllm":
    VLLM_URL, VLLM_KEY, VLLM_MODEL = configure_vllm_client_env(
        base_url=os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1"),
        api_key=os.environ.get("OPENAI_API_KEY", "abc-123"),
        model=os.environ.get("VLLM_MODEL", "Qwen3-30B-A3B"),
    )
    print("vLLM client config:", VLLM_URL, VLLM_MODEL)
    if not verify_vllm_server(VLLM_URL, VLLM_KEY):
        print("Start vLLM in a terminal (section 9a) then re-run this cell.")
    else:
        pr_llm_results = run_llm_summaries_for_demo_prs(
            backend="vllm",
            base_url=VLLM_URL,
            model=VLLM_MODEL,
        )
else:
    pr_llm_results = run_llm_summaries_for_demo_prs(
        backend="openai",
        model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
    )

ok = [pr for pr, text in pr_llm_results.items() if text]
if ok:
    print(f"\nLLM summaries written for PRs: {', '.join(f'#{p}' for p in ok)}")
    print("Files: out/pr-<N>/executive_summary_llm.md")
elif LLM_BACKEND == "vllm":
    print(
        "No LLM summaries produced. For vLLM: complete section 9a, then re-run. "
        "For OpenAI: set SUMMARY_LLM_BACKEND=openai and OPENAI_API_KEY in .env."
    )
else:
    print("No LLM summaries produced. Set OPENAI_API_KEY in .env and re-run.")

## 10. Demo checklist

- [ ] `pytest` green
- [ ] PR #5572 → `test:miopen`, `test_filter:quick`, topology warnings show (none expected)
- [ ] Section 9 → `executive_summary_llm.md` for each demo PR when vLLM/OpenAI available
- [ ] PR #5480 → `test_filter:quick` (third-party packaging)
- [ ] Executive summary + HTML include **Topology warnings** section
- [ ] Open `out/pr-*/report.html` in browser
- [ ] Fork Actions: **Change Impact Upstream PR Scan** workflow dispatch